# Практическое занятие: Настройка нейронов и сравнение активаций

**Цель работы**: Научиться подбирать веса и смещения нейронов, понять влияние функции активации на классификацию, а также убедиться, что двухслойная сеть может решать нелинейные задачи.

**План**:
1. Импорт библиотек и вспомогательные функции.
2. Задание 1: Однослойный нейрон для OR.
3. Задание 2: Сравнение функций активации на задаче AND.
4. Задание 3: Двухслойная сеть для прямоугольной области.
5. Анализ результатов.

---

In [1]:
# Импортируем всё необходимое
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, HBox, VBox, widgets
import networkx as nx

In [2]:
# Функции активации и их производные (для справки)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def step(x):
    return (x >= 0).astype(int)

## Задание 1. Один нейрон для OR

Напомним таблицу истинности OR:

| x1 | x2 | y |
|----|----|---|
| 0  | 0  | 0 |
| 0  | 1  | 1 |
| 1  | 0  | 1 |
| 1  | 1  | 1 |

Используя ступенчатую функцию активации, подберите вес $w_1$, $w_2$ и смещение $b$ так, чтобы нейрон правильно классифицировал все четыре точки: на графике ниже синие точки должны оказаться в синей области, а красные точки - в красной области.

**Инструкция**: двигайте ползунки и следите за точностью. При точности 100% вы нашли решение.
Обязательно перед отправкой (публикацией) блокнота убедитесь, что ваш выбор в ползунках и на графике сохраняется. Если нет - напишите прямо текстом ответ.

In [3]:
# Данные для OR
X_or = np.array([[0,0],[0,1],[1,0],[1,1]])
y_or = np.array([0,1,1,1])

def plot_or_neuron(w1, w2, b):
    # Сетка для визуализации
    xx, yy = np.meshgrid(np.linspace(-0.5,1.5,200),
                         np.linspace(-0.5,1.5,200))
    z = w1*xx + w2*yy + b
    a = step(z)

    plt.figure(figsize=(12,4))

    # 1. Карта классификации
    plt.subplot(1,2,1)
    plt.contourf(xx, yy, a, levels=[-0.5,0.5,1.5], colors=['blue','red'], alpha=0.5)
    plt.scatter(X_or[y_or==0,0], X_or[y_or==0,1], color='blue', s=200, label='0', edgecolor='k')
    plt.scatter(X_or[y_or==1,0], X_or[y_or==1,1], color='red', s=200, label='1', edgecolor='k')
    plt.xlim(-0.5,1.5)
    plt.ylim(-0.5,1.5)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('Классификация нейроном')
    plt.legend()
    plt.grid(True)

    # 2. Точность
    z_vals = w1*X_or[:,0] + w2*X_or[:,1] + b
    pred = step(z_vals)
    acc = np.mean(pred == y_or)

    plt.subplot(1,2,2)
    plt.axis('off')
    plt.text(0.5,0.5, f'Точность: {acc*100:.1f}%', fontsize=20, ha='center')
    if acc == 1.0:
        plt.text(0.5,0.3, '✓ Решение найдено!', fontsize=16, color='green', ha='center')
    else:
        plt.text(0.5,0.3, 'Продолжайте подбор', fontsize=16, color='gray', ha='center')

    plt.show()

interact(plot_or_neuron,
         w1=FloatSlider(min=-2, max=2, step=0.1, value=0, description='w1'),
         w2=FloatSlider(min=-2, max=2, step=0.1, value=0, description='w2'),
         b=FloatSlider(min=-2, max=2, step=0.1, value=0, description='b'));

interactive(children=(FloatSlider(value=0.0, description='w1', max=2.0, min=-2.0), FloatSlider(value=0.0, desc…

**Вопрос**: Почему выбранные вами значения работают? Что произойдёт, если сделать смещение очень большим отрицательным числом?


w1= 1.0, w2 = 1.0 b = -0.7
потому что линия x2*1.0 =-x1*1.0 + 0.7 разделяет эти точки между собой

если сделать слишком большое отрицательное смещение, то чтобы разделить все точки правильно предётся пропорционально увеличивать w1 и w2

## Задание 2. Сравнение функций активации с численными метриками

Теперь посмотрим, как разные функции активации влияют на поведение нейрона. Возьмём задачу AND (таблица ниже) и один нейрон. Будем использовать три активации: сигмоиду, гиперболический тангенс (tanh) и ReLU. Для каждой активации построим карту отклика, столбчатую диаграмму значений на точках и вычислим метрики:
- **Точность** (если порог 0.5)
- **Средняя активация для класса 0** (чем ближе к 0, тем лучше)
- **Средняя активация для класса 1** (чем ближе к 1, тем лучше)
- **Разность средних** (margin, чем больше, тем лучше разделение)
- **Среднеквадратичная ошибка (MSE)** между выходом и целевым значением

**Таблица AND**:
| x1 | x2 | y |
|----|----|---|
| 0  | 0  | 0 |
| 0  | 1  | 0 |
| 1  | 0  | 0 |
| 1  | 1  | 1 |

Подберите параметры так, чтобы нейрон правильно классифицировал точки, и сравните метрики для разных активаций.

In [ ]:
# Данные для AND
X_and = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0,0,0,1])

def compute_metrics(activation_fn, X, y, w1, w2, b, scale_to_01=False):
    z = w1*X[:,0] + w2*X[:,1] + b
    a = activation_fn(z)
    if scale_to_01:  # для tanh масштабируем в [0,1] для единообразия метрик
        a = (a + 1)/2
    # Точность при пороге 0.5
    pred = (a >= 0.5).astype(int)
    acc = np.mean(pred == y)
    # Средние для классов
    mean_0 = np.mean(a[y==0])
    mean_1 = np.mean(a[y==1])
    margin = mean_1 - mean_0
    # MSE
    mse = np.mean((a - y)**2)
    return acc, mean_0, mean_1, margin, mse, a

def compare_activations_with_metrics(w1, w2, b):
    # Вычисляем выход для разных активаций на точках данных и на сетке
    xx, yy = np.meshgrid(np.linspace(-0.5,1.5,200),
                         np.linspace(-0.5,1.5,200))
    z_grid = w1*xx + w2*yy + b

    # Сигмоида
    a_sig_grid = sigmoid(z_grid)
    acc_sig, m0_sig, m1_sig, marg_sig, mse_sig, a_sig = compute_metrics(sigmoid, X_and, y_and, w1, w2, b)

    # Tanh (масштабируем в [0,1] для сравнения)
    a_tanh_grid = (tanh(z_grid) + 1)/2
    acc_tanh, m0_tanh, m1_tanh, marg_tanh, mse_tanh, a_tanh = compute_metrics(tanh, X_and, y_and, w1, w2, b, scale_to_01=True)

    # ReLU (для метрик используем исходный, для сетки нормируем для наглядности)
    a_relu_grid = relu(z_grid)
    a_relu_grid_norm = a_relu_grid / (np.max(a_relu_grid) + 1e-8)
    acc_relu, m0_relu, m1_relu, marg_relu, mse_relu, a_relu = compute_metrics(relu, X_and, y_and, w1, w2, b, scale_to_01=False)

    # Создаём фигуру с 3 строками и 3 столбцами
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))

    # Строка 1: Сигмоида
    # Карта
    ax = axes[0,0]
    im = ax.contourf(xx, yy, a_sig_grid, levels=20, cmap='RdBu', alpha=0.7)
    plt.colorbar(im, ax=ax)
    ax.scatter(X_and[y_and==0,0], X_and[y_and==0,1], color='blue', s=100, label='0', edgecolor='k')
    ax.scatter(X_and[y_and==1,0], X_and[y_and==1,1], color='red', s=100, label='1', edgecolor='k')
    ax.set_xlim(-0.5,1.5); ax.set_ylim(-0.5,1.5)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title('Сигмоида')
    ax.grid(True)

    # Столбцы
    ax = axes[0,1]
    colors = ['blue' if y == 0 else 'red' for y in y_and]
    ax.bar(range(4), a_sig, color=colors, edgecolor='k')
    ax.axhline(0.5, color='gray', linestyle='--')
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'({x[0]},{x[1]})' for x in X_and], rotation=45)
    ax.set_ylabel('Выход')
    ax.set_title('Значения на точках')
    ax.set_ylim(-0.1, 1.1)

    # Метрики
    ax = axes[0,2]
    ax.axis('off')
    metrics_text = f"""Сигмоида:
    Точность: {acc_sig*100:.1f}%
    Среднее (класс 0): {m0_sig:.3f}
    Среднее (класс 1): {m1_sig:.3f}
    Разность средних: {marg_sig:.3f}
    MSE: {mse_sig:.3f}"""
    ax.text(0.1, 0.5, metrics_text, fontsize=12, fontfamily='monospace', verticalalignment='center')

    # Строка 2: Tanh
    ax = axes[1,0]
    im = ax.contourf(xx, yy, a_tanh_grid, levels=20, cmap='RdBu', alpha=0.7)
    plt.colorbar(im, ax=ax)
    ax.scatter(X_and[y_and==0,0], X_and[y_and==0,1], color='blue', s=100, label='0', edgecolor='k')
    ax.scatter(X_and[y_and==1,0], X_and[y_and==1,1], color='red', s=100, label='1', edgecolor='k')
    ax.set_xlim(-0.5,1.5); ax.set_ylim(-0.5,1.5)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title('Tanh (масштабированный в [0,1])')
    ax.grid(True)

    ax = axes[1,1]
    colors = ['blue' if y == 0 else 'red' for y in y_and]
    ax.bar(range(4), a_tanh, color=colors, edgecolor='k')
    ax.axhline(0.5, color='gray', linestyle='--')
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'({x[0]},{x[1]})' for x in X_and], rotation=45)
    ax.set_ylabel('Выход')
    ax.set_title('Значения на точках')
    ax.set_ylim(-0.1, 1.1)

    ax = axes[1,2]
    ax.axis('off')
    metrics_text = f"""Tanh (масшт.):
    Точность: {acc_tanh*100:.1f}%
    Среднее (класс 0): {m0_tanh:.3f}
    Среднее (класс 1): {m1_tanh:.3f}
    Разность средних: {marg_tanh:.3f}
    MSE: {mse_tanh:.3f}"""
    ax.text(0.1, 0.5, metrics_text, fontsize=12, fontfamily='monospace', verticalalignment='center')

    # Строка 3: ReLU
    ax = axes[2,0]
    im = ax.contourf(xx, yy, a_relu_grid_norm, levels=20, cmap='RdBu', alpha=0.7)
    plt.colorbar(im, ax=ax)
    ax.scatter(X_and[y_and==0,0], X_and[y_and==0,1], color='blue', s=100, label='0', edgecolor='k')
    ax.scatter(X_and[y_and==1,0], X_and[y_and==1,1], color='red', s=100, label='1', edgecolor='k')
    ax.set_xlim(-0.5,1.5); ax.set_ylim(-0.5,1.5)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title('ReLU (нормированная карта)')
    ax.grid(True)

    ax = axes[2,1]
    colors = ['blue' if y == 0 else 'red' for y in y_and]
    ax.bar(range(4), a_relu, color=colors, edgecolor='k')
    ax.axhline(0.5, color='gray', linestyle='--')
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'({x[0]},{x[1]})' for x in X_and], rotation=45)
    ax.set_ylabel('Выход (исходный)')
    ax.set_title('Значения на точках (ReLU)')
    ax.set_ylim(min(-0.1, a_relu.min()-0.1), max(1.5, a_relu.max()+0.1))

    ax = axes[2,2]
    ax.axis('off')
    metrics_text = f"""ReLU (исходный):
    Точность: {acc_relu*100:.1f}%
    Среднее (класс 0): {m0_relu:.3f}
    Среднее (класс 1): {m1_relu:.3f}
    Разность средних: {marg_relu:.3f}
    MSE: {mse_relu:.3f}"""
    ax.text(0.1, 0.5, metrics_text, fontsize=12, fontfamily='monospace', verticalalignment='center')

    plt.tight_layout()
    plt.show()

interact(compare_activations_with_metrics,
         w1=FloatSlider(min=-2, max=2, step=0.1, value=1.0, description='w1'),
         w2=FloatSlider(min=-2, max=2, step=0.1, value=1.0, description='w2'),
         b=FloatSlider(min=-2, max=2, step=0.1, value=-1.5, description='b'));

interactive(children=(FloatSlider(value=1.0, description='w1', max=2.0, min=-2.0), FloatSlider(value=1.0, desc…

**Вопросы**:
1. Какая активация даёт наибольшую разность средних (margin)? Что это означает?
2. Почему у ReLU среднее для класса 0 может быть не нулевым?
3. Как изменение смещения влияет на MSE?

1) Наибольшую разность даёт ReLU, это значит, что среди рассматриваемых фукнций, ей можно точнее всего разделить классы
2) У ReLU среднее для класса 0 может быть не нулевым, потому что функция ReLU (max(0, z)) возвращает положительные значения, если линейная комбинация входов z = w1x1 + w2x2 + b оказывается положительной
3) Если увеличивается, граница смещения сдвиается что может улучшить или ухудшить точность классификации

## Задание 3. Двухслойная сеть для прямоугольной области

Рассмотрим задачу, где класс 1 — это точки, у которых обе координаты больше 0.3, а класс 0 — все остальные. Для наглядности возьмём девять точек с координатами 0, 0.5 и 1:

- (0,0): 0
- (0,0.5): 0
- (0,1): 0
- (0.5,0): 0
- (0.5,0.5): 1
- (0.5,1): 1
- (1,0): 0
- (1,0.5): 1
- (1,1): 1

Эта задача не решается одним нейроном (почему?), но двухслойная сеть с двумя нейронами в скрытом слое справится. Один нейрон должен определять условие $x_1 > 0.3$, второй — $x_2 > 0.3$, а выходной нейрон — логическое AND этих двух условий.

Используя ступенчатую активацию, подберите веса и смещения так, чтобы сеть правильно классифицировала все девять точек.

**Перед отправкой блокнота убедитесь, что положения слайдеров и графики сохранились, иначе сделайте снимок результирующего состояния.**


In [ ]:
# Данные для прямоугольной области
X_rect = np.array([[0,0], [0,0.5], [0,1], [0.5,0], [0.5,0.5], [0.5,1], [1,0], [1,0.5], [1,1]])
y_rect = np.array([0,0,0,0,1,1,0,1,1])

def forward_rect(X, w1, b1, w2, b2, wo, bo):
    # X: (N,2)
    z1 = X[:,0]*w1[0] + X[:,1]*w1[1] + b1
    a1 = step(z1)
    z2 = X[:,0]*w2[0] + X[:,1]*w2[1] + b2
    a2 = step(z2)
    z_out = a1*wo[0] + a2*wo[1] + bo
    y = step(z_out)
    return y, a1, a2

def plot_rect_network(w11, w12, b1, w21, w22, b2, wo1, wo2, bo):
    w1 = np.array([w11, w12])
    w2 = np.array([w21, w22])
    wo = np.array([wo1, wo2])

    # Предсказание
    y_pred, a1, a2 = forward_rect(X_rect, w1, b1, w2, b2, wo, bo)
    acc = np.mean(y_pred == y_rect)

    # Сетка для визуализации
    xx, yy = np.meshgrid(np.linspace(-0.2,1.2,200),
                         np.linspace(-0.2,1.2,200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    y_grid, _, _ = forward_rect(grid, w1, b1, w2, b2, wo, bo)
    y_grid = y_grid.reshape(xx.shape)

    plt.figure(figsize=(15,5))

    # Схема сети
    plt.subplot(1,3,1)
    G = nx.DiGraph()
    G.add_node('x1', pos=(0,2))
    G.add_node('x2', pos=(0,0))
    G.add_node('h1', pos=(2,3))
    G.add_node('h2', pos=(2,-1))
    G.add_node('y', pos=(4,1))
    edges = [('x1','h1'),('x2','h1'),('x1','h2'),('x2','h2'),('h1','y'),('h2','y')]
    G.add_edges_from(edges)
    pos = nx.get_node_attributes(G,'pos')
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=2000, edgecolors='k')
    nx.draw_networkx_labels(G, pos, labels={'x1':'x₁','x2':'x₂','h1':'h₁','h2':'h₂','y':'y'}, font_size=12)
    nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, edge_color='gray', width=1.5)
    edge_labels = {('x1','h1'):f'{w1[0]:.2f}', ('x2','h1'):f'{w1[1]:.2f}',
                   ('x1','h2'):f'{w2[0]:.2f}', ('x2','h2'):f'{w2[1]:.2f}',
                   ('h1','y'):f'{wo[0]:.2f}', ('h2','y'):f'{wo[1]:.2f}'}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9)
    plt.text(2,3.5, f'b₁={b1:.2f}', fontsize=9, ha='center', color='red')
    plt.text(2,-1.5, f'b₂={b2:.2f}', fontsize=9, ha='center', color='red')
    plt.text(4,1.5, f'bₒ={bo:.2f}', fontsize=9, ha='center', color='red')
    plt.title('Схема сети')
    plt.axis('off')

    # Выход сети
    plt.subplot(1,3,2)
    plt.contourf(xx, yy, y_grid, levels=[-0.5,0.5,1.5], colors=['blue','red'], alpha=0.5)
    plt.scatter(X_rect[y_rect==0,0], X_rect[y_rect==0,1], color='blue', s=100, label='0', edgecolor='k')
    plt.scatter(X_rect[y_rect==1,0], X_rect[y_rect==1,1], color='red', s=100, label='1', edgecolor='k')
    plt.xlim(-0.2,1.2)
    plt.ylim(-0.2,1.2)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('Классификация сетью')
    plt.legend()
    plt.grid(True)

    # Точность
    plt.subplot(1,3,3)
    plt.axis('off')
    plt.text(0.5,0.5, f'Точность: {acc*100:.1f}%', fontsize=20, ha='center')
    if acc == 1.0:
        plt.text(0.5,0.3, '✓ Решение найдено!', fontsize=16, color='green', ha='center')
    else:
        plt.text(0.5,0.3, 'Продолжайте подбор', fontsize=16, color='gray', ha='center')

    plt.tight_layout()
    plt.show()

# Создаём ползунки
style = {'description_width':'initial'}
slider_kw = dict(min=-2, max=2, step=0.1, style=style)

w11 = FloatSlider(value=0, description='w₁₁ (h1,x1)', **slider_kw)
w12 = FloatSlider(value=0, description='w₁₂ (h1,x2)', **slider_kw)
b1  = FloatSlider(value=0, description='b₁ (h1)', **slider_kw)

w21 = FloatSlider(value=0, description='w₂₁ (h2,x1)', **slider_kw)
w22 = FloatSlider(value=0, description='w₂₂ (h2,x2)', **slider_kw)
b2  = FloatSlider(value=0, description='b₂ (h2)', **slider_kw)

wo1 = FloatSlider(value=0, description='wₒ₁ (h1→y)', **slider_kw)
wo2 = FloatSlider(value=0, description='wₒ₂ (h2→y)', **slider_kw)
bo  = FloatSlider(value=0, description='bₒ (out)', **slider_kw)

ui = VBox([
    HBox([widgets.Label('Нейрон h₁ (условие x₁>0.3)'), widgets.Label('Нейрон h₂ (условие x₂>0.3)'), widgets.Label('Выходной нейрон (AND)')]),
    HBox([
        VBox([w11, w12, b1]),
        VBox([w21, w22, b2]),
        VBox([wo1, wo2, bo])
    ])
])

out = widgets.interactive_output(plot_rect_network, {
    'w11': w11, 'w12': w12, 'b1': b1,
    'w21': w21, 'w22': w22, 'b2': b2,
    'wo1': wo1, 'wo2': wo2, 'bo': bo
})

display(ui, out)

Output()

w11 = 0;w12=1;b1=-0.3;w21=1;w22=0;b2=-0.3;wh1=1;wh2=1;bo=-1.3
**Вопросы для анализа**:
1. Почему один нейрон не может решить эту задачу?
2. Какие функции реализуют скрытые нейроны в правильном решении?
3. Что произойдёт, если изменить вес выходного нейрона на отрицательный?

Один нейрон может не может разделить классы, только одной линией
они реализуют функции h2 = step(x2-0.3) и h1 = step(x1 - 0.3)
если изменить веса выходного нейрона и смещение на противоположные, то классификация с точносьтю до наоборот будет выдввать результат

## Заключение

Вы выполнили три задания:
- Настроили нейрон для OR.
- Сравнили поведение разных функций активации на задаче AND.
- Построили двухслойную сеть для нелинейной задачи (прямоугольная область).

Эти упражнения закладывают основу для понимания многослойных сетей и алгоритмов их обучения. Спасибо за работу!